In [ ]:
# Import necessary libraries
import subprocess
import numpy as np
import matplotlib.pyplot as plt
import os
import gmsh
from pathlib import Path
from jinja2 import Template
import pyvista as pv

In [ ]:
class Generator:
    def __init__(self, domain_size, n_elements, generator_type, CNN_params=None):
        self.domain_length = domain_size[0]
        self.domain_height = domain_size[1]
        self.n_elements_x = n_elements[0]
        self.n_elements_y = n_elements[1]
        self.generator_type = generator_type
        self.mesh_min = min(self.domain_length / self.n_elements_x, self.domain_height / self.n_elements_y)
        self.mesh_max = max(self.domain_length / self.n_elements_x, self.domain_height / self.n_elements_y)
        self.mesh_params = {'mesh_algorithm': 8, 'mesh_recombine': 1, 'mesh_element_order': 2}
        self.geo_generator = self.initialize_generator(generator_type, CNN_params)

        def initialize_generator(self, type, params):
            if type == 'CNN' and params is not None:
                # if CNN weights and biases are provided, we can initialize a CNN generator with those parameters
                return CNNGenerator(params)
            elif type == 'CNN':
                # if CNN weights and biases are not provided, we can initialize a default CNN generator
                return CNNGenerator()
            else:    
                # return a basic geometry generator
                return None
            
        def create_custom_hx(self, points):
            # create a custom hx obstruction in gmsh using the provided points
            n_points = len(points)
            gmsh_points = []
            for coord in points:
                point = gmsh.model.occ.addPoint(coord[0], coord[1], 0)
                gmsh_points.append(point)

            # Connect the points with lines to form the geometry of the obstruction
            lines = []
            for i in range(n_points):
                p1 = points[i]
                p2 = points[(i + 1) % n_points]  # Wrap around to connect the last point to the first
                line = gmsh.model.occ.addLine(p1, p2)
                lines.append(line)
            
            # create a surface from the lines
            cl = gmsh.model.occ.addCurveLoop(lines)
            surface = gmsh.model.occ.addPlaneSurface([cl])
            return surface
        
        def generate_mesh(self):
            # Initialize gmsh and create a new model
            gmsh.initialize()
            gmsh.model.add("heat_exchanger")

            # Use OpenCASCADE geometry kernel
            gmsh.model.occ

            # Define the domain geometry
            rect = gmsh.model.occ.addRectangle(0, -self.domain_height/2, 0, self.domain_length, self.domain_height)
            gmsh.model.occ.synchronize()

            # Add obstructions to the geometry based on the hx_objects provided by the generator
            obstructions = []
            for obj in self.hx_objects:
                obstruction = self.create_custom_hx(obj)
                obstructions.append(obstruction)
            gmsh.model.occ.synchronize()

            # Boolean cut
            cut = gmsh.model.occ.cut(
                [(2, rect)],
                [(2, s) for s in obstructions],
                removeObject=True,
                removeTool=True
            )
            gmsh.model.occ.synchronize()

            # Extract the resulting geometry after the cut operation
            outDimTags, _ = cut
            fluid_surfaces = [tag for dim, tag in outDimTags if dim == 2]

            if not fluid_surfaces:
                raise RuntimeError("Boolean cut resulted in no fluid domain. Please check the geometry of the obstructions.")
            
            gmsh.model.addPhysicalGroup(2, fluid_surfaces, name="Fluid")

            print("Cut result:", outDimTags)
            print("2D entities:", gmsh.model.getEntities(2))

            # Cylinder wall curves
            gmsh.model.occ.synchronize()
            boundaries = gmsh.model.getBoundary(
                [(2, s) for s in fluid_surfaces],
                oriented=False
            )

            obstruction_wall_curves = []
            inlet = []
            outlet = []
            top = []
            bottom = []

            for dim, tag in boundaries:
                com = gmsh.model.occ.getCenterOfMass(dim, tag)
                x, y, _ = com

                if abs(x) < 1e-6:
                    inlet.append(tag)
                elif abs(x - self.length) < 1e-6:
                    outlet.append(tag)
                elif abs(y - self.domain_height/2) < 1e-6:
                    top.append(tag)
                elif abs(y + self.domain_height/2) < 1e-6:
                    bottom.append(tag)
                else:
                    obstruction_wall_curves.append(tag)
            
            gmsh.model.addPhysicalGroup(1, obstruction_wall_curves, name="Wall")
            gmsh.model.addPhysicalGroup(1, inlet, name="Inlet")
            gmsh.model.addPhysicalGroup(1, outlet, name="Outlet")
            gmsh.model.addPhysicalGroup(1, top, name="Top")
            gmsh.model.addPhysicalGroup(1, bottom, name="Bottom")
            
            # Set mesh parameters
            gmsh.option.setNumber("Mesh.CharacteristicLengthMin", self.mesh_params['mesh_min'])
            gmsh.option.setNumber("Mesh.CharacteristicLengthMax", self.mesh_params['mesh_max'])
            gmsh.option.setNumber("Mesh.Algorithm", self.mesh_params['mesh_algorithm']) # 8 
            gmsh.option.setNumber("Mesh.RecombineAll", self.mesh_params['mesh_recombine']) # 1
            gmsh.option.setNumber("Mesh.ElementOrder", self.mesh_params['mesh_element_order']) # 2

            # Generate the mesh
            gmsh.model.mesh.generate(2)
            gmsh.write(self.filename)
            gmsh.finalize()

In [ ]:
class Simulator:
    def __init__(self, type='MOOSE', params="./cutthroat-opt"):
        self.type = type
        if self.type == 'MOOSE':
            self.app_name = params
        else:
            pass

    def run_moose(self, input_file, n_processors=1):
        '''Run a MOOSE simulation inside a subprocess
        
        Parameters:
        input_file: str, path to the MOOSE input file (.i)
        n_processors: int, number of processors to use for the simulation
        '''
        # Construct the command to run MOOSE
        cmd = [
            "conda", "run", "-n", "moose",
            "mpiexec", "-np", str(n_processors),
            self.app_name, "-i", input_file
        ]

        # Run the command and capture the output
        result = subprocess.run(cmd, capture_output=True, text=True)

        # Check for errors
        if result.returncode != 0:
            print("Error running MOOSE:")
            print(result.stderr)
        else:
            print("MOOSE simulation completed successfully.")
            print(result.stdout)

    def analyze_exodus_results(self, exodus_file):
        # Placeholder for analyzing Exodus results
        # This function would read the Exodus file, extract relevant data, and perform analysis
        mesh = pv.read(exodus_file)
        print(mesh)
        print(mesh.array_names)
    
    def run(self, input_file, n_processors=4):
        if self.type == 'MOOSE':
            self.run_moose(input_file, n_processors)
        else:
            pass

In [ ]:
class Optimizer:
    def __init__(self, domain_dimensions, generator_type, simulator_type, mesh_files_dir, results_dir):
        self.generator = self.initialize_generator(generator_type, domain_dimensions)
        self.simulator = self.initialize_simulator(simulator_type)
        self.mesh_files_dir = mesh_files_dir
        self.results_dir = results_dir

    def initialize_generator(self, generator_type, domain_dimensions):
        gen = Generator(domain_size=domain_dimensions[0], n_elements=domain_dimensions[1], generator_type=generator_type)
        return gen
    
    def initialize_simulator(self, simulator_type):
        sim = Simulator(type=simulator_type)
        return sim

    def create_input_file(self, template_path, output_dir, mesh_name):
        '''Generate a MOOSE .i file from a template
        
        Parameters:
        - template_path: path to template.i file
        - output_dir: directory to save the generated .i file
        - mesh_name: name of the mesh file to read e.g. 'rd0001.msh'
        '''
        # Extract case prefix and number from mesh_name (assumes format 'rdXXXX.msh' or 'gdXXXX.msh')
        case_prefix = mesh_name.split('.')[0][:-4]
        case_number = mesh_name.split('.')[0][-4:]
        case_id = f"{case_prefix}{case_number}"

        input_filename = f"{case_id}.i"

        # Load the template
        with open(template_path, "r") as f:
            template = Template(f.read())
        
        # Render the template with the mesh name
        rendered = template.render(
            mesh_name=mesh_name,
            case_name=case_id,
            rho=1.0,  # Placeholder value for density
            mu=1.0,   # Placeholder value for viscosity
            k=1.0,    # Placeholder value for thermal conductivity
            cp=1.0    # Placeholder value for specific heat capacity
        )

        # Ensure output directory exists
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

        # Write file to output directory
        output_path = output_dir / input_filename
        with open(output_path, "w") as f:
            f.write(rendered)
        
        return output_path

    def clean_input_file(self, input_file):
        '''Delete the generated .i file after simulation'''
        if os.path.exists(input_file):
            os.remove(input_file)

    def run_simulation(self, input_file, n_processors=None):
        '''Run the simulation using the optimizer's simulator
        
        Parameters:
        - input_file: path to the .i file to run
        - n_processors: number of processors to use for parallel execution if using MOOSE simulator (optional)
        '''
        if n_processors is not None:
            self.simulator.run(input_file, n_processors)
        else:
            self.simulator.run(input_file)

    def simulate_all_meshes(self):
        '''Simulate all meshes in the mesh_files_dir'''
        mesh_files = [f for f in os.listdir(self.mesh_files_dir) if f.endswith('.msh')]
        for mesh_file in mesh_files:
            print(f"Simulating for mesh: {mesh_file}")
            input_file = self.create_input_file(
                template_path="problems/template.i",
                output_dir=self.results_dir,
                mesh_name=mesh_file
            )
            self.run_simulation(input_file, n_processors=4)
            self.clean_input_file(input_file)
        